# Silver Layer v2 — Rebuild from Bronze
Full rebuild: data profiling first, then transformations. Produces `silver_train`, `silver_test`, and a new `silver_store` (one row per store, cleanly imputed) — the last version skipped this and downstream models fell back to raw `bronze_store` as a result.

## 1. Load bronze

In [0]:
train_df = spark.table("retail_intelligence.bronze.train")
test_df = spark.table("retail_intelligence.bronze.test")
store_df = spark.table("retail_intelligence.bronze.store")

print("Train:", train_df.count(), "| Test:", test_df.count(), "| Store:", store_df.count())

Train: 1017209 | Test: 41088 | Store: 1115


## 2. Data profiling (before any changes)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count

def null_profile(df, name):
    print(f"--- {name} null counts ---")
    df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

null_profile(train_df, "train")
null_profile(test_df, "test")
null_profile(store_df, "store")

--- train null counts ---
+-----+---------+----+-----+---------+----+-----+------------+-------------+------------+
|Store|DayOfWeek|Date|Sales|Customers|Open|Promo|StateHoliday|SchoolHoliday|_ingested_at|
+-----+---------+----+-----+---------+----+-----+------------+-------------+------------+
|    0|        0|   0|    0|        0|   0|    0|           0|            0|           0|
+-----+---------+----+-----+---------+----+-----+------------+-------------+------------+

--- test null counts ---
+---+-----+---------+----+----+-----+------------+-------------+------------+
| Id|Store|DayOfWeek|Date|Open|Promo|StateHoliday|SchoolHoliday|_ingested_at|
+---+-----+---------+----+----+-----+------------+-------------+------------+
|  0|    0|        0|   0|  11|    0|           0|            0|           0|
+---+-----+---------+----+----+-----+------------+-------------+------------+

--- store null counts ---
+-----+---------+----------+-------------------+-------------------------+-------

In [0]:
# Duplicate checks
print("Duplicate (Store, Date) in train:")
train_df.groupBy("Store", "Date").agg(count("*").alias("c")).filter("c > 1").show()

print("Duplicate Store in store table:")
store_df.groupBy("Store").agg(count("*").alias("c")).filter("c > 1").show()

Duplicate (Store, Date) in train:
+-----+----+---+
|Store|Date|  c|
+-----+----+---+
+-----+----+---+

Duplicate Store in store table:
+-----+---+
|Store|  c|
+-----+---+
+-----+---+



In [0]:
# Distinct value profiling for categoricals
for c in ["StoreType", "Assortment", "PromoInterval"]:
    print(f"--- {c} ---")
    store_df.select(c).distinct().show(20, False)

train_df.select("StateHoliday").distinct().show()

--- StoreType ---
+---------+
|StoreType|
+---------+
|c        |
|a        |
|d        |
|b        |
+---------+

--- Assortment ---
+----------+
|Assortment|
+----------+
|a         |
|c         |
|b         |
+----------+

--- PromoInterval ---
+----------------+
|PromoInterval   |
+----------------+
|Jan,Apr,Jul,Oct |
|Feb,May,Aug,Nov |
|Mar,Jun,Sept,Dec|
|NULL            |
+----------------+

+------------+
|StateHoliday|
+------------+
|           0|
|           a|
|           b|
|           c|
+------------+



In [0]:
# Numeric distribution sanity check
store_df.describe(["CompetitionDistance", "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"]).show()
train_df.describe(["Sales", "Customers"]).show()

+-------+-------------------+-------------------------+------------------------+
|summary|CompetitionDistance|CompetitionOpenSinceMonth|CompetitionOpenSinceYear|
+-------+-------------------+-------------------------+------------------------+
|  count|               1112|                      761|                     761|
|   mean|  5404.901079136691|       7.2247043363994745|      2008.6688567674114|
| stddev|  7663.174720367948|       3.2123477966147087|       6.195982559329067|
|    min|                 20|                        1|                    1900|
|    max|              75860|                       12|                    2015|
+-------+-------------------+-------------------------+------------------------+

+-------+-----------------+------------------+
|summary|            Sales|         Customers|
+-------+-----------------+------------------+
|  count|          1017209|           1017209|
|   mean|5773.818972305593| 633.1459464082602|
| stddev|3849.926175234757|464.4117

## 3. Fix `Open` nulls in test
Sunday (`DayOfWeek == 7`) with unknown Open status → assume closed. Everything else → assume open.

In [0]:
from pyspark.sql.functions import when

test_df = test_df.withColumn(
    "Open",
    when(col("Open").isNull() & (col("DayOfWeek") == 7), 0)
    .when(col("Open").isNull(), 1)
    .otherwise(col("Open"))
)
print("Remaining Open nulls:", test_df.filter(col("Open").isNull()).count())

Remaining Open nulls: 0


## 4. Build clean `silver_store` (one row per store)
This is the table other layers should join against — not raw `bronze_store`.

In [0]:
# Flag BEFORE filling: preserves "no known competitor" as distinct from "average competitor"
silver_store_df = store_df.withColumn(
    "HasCompetitionInfo",
    when(col("CompetitionDistance").isNotNull(), 1).otherwise(0)
)

month_median = store_df.approxQuantile("CompetitionOpenSinceMonth", [0.5], 0.01)[0]
year_median = store_df.approxQuantile("CompetitionOpenSinceYear", [0.5], 0.01)[0]
distance_max = store_df.approxQuantile("CompetitionDistance", [1.0], 0.01)[0]

silver_store_df = silver_store_df.fillna({
    "CompetitionDistance": distance_max,
    "CompetitionOpenSinceMonth": month_median,
    "CompetitionOpenSinceYear": year_median,
})

print("Distance fill (max):", distance_max, "| Month median:", month_median, "| Year median:", year_median)
silver_store_df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in silver_store_df.columns]).show()

Distance fill (max): 75860.0 | Month median: 7.0 | Year median: 2009.0
+-----+---------+----------+-------------------+-------------------------+------------------------+------+---------------+---------------+-------------+------------+------------------+
|Store|StoreType|Assortment|CompetitionDistance|CompetitionOpenSinceMonth|CompetitionOpenSinceYear|Promo2|Promo2SinceWeek|Promo2SinceYear|PromoInterval|_ingested_at|HasCompetitionInfo|
+-----+---------+----------+-------------------+-------------------------+------------------------+------+---------------+---------------+-------------+------------+------------------+
|    0|        0|         0|                  0|                        0|                       0|     0|            544|            544|          544|           0|                 0|
+-----+---------+----------+-------------------+-------------------------+------------------------+------+---------------+---------------+-------------+------------+------------------+



## 5. Join store attributes into train/test

In [0]:
silver_train_df = train_df.join(silver_store_df, on="Store", how="left")
silver_test_df = test_df.join(silver_store_df, on="Store", how="left")

print("Train rows:", silver_train_df.count(), "| Test rows:", silver_test_df.count())

Train rows: 1017209 | Test rows: 41088


## 6. Date features

In [0]:
from pyspark.sql.functions import year, month, weekofyear, dayofmonth, quarter

for name in ["silver_train_df", "silver_test_df"]:
    df = globals()[name]
    df = (
        df.withColumn("Year", year("Date"))
          .withColumn("Month", month("Date"))
          .withColumn("Week", weekofyear("Date"))
          .withColumn("Day", dayofmonth("Date"))
          .withColumn("Quarter", quarter("Date"))
          .withColumn("IsWeekend", when(col("DayOfWeek").isin(6, 7), 1).otherwise(0))
    )
    globals()[name] = df

## 7. Promo2 activity flag (real per-row promo status, not just the static flag)

In [0]:
from pyspark.sql.functions import date_format, instr

for name in ["silver_train_df", "silver_test_df"]:
    df = globals()[name]
    df = df.withColumn(
        "IsPromo2Active",
        when(
            (col("Promo2") == 1) & (instr(col("PromoInterval"), date_format(col("Date"), "MMM")) > 0),
            1
        ).otherwise(0)
    )
    globals()[name] = df

## 8. Competition age (months), clipped at 0

In [0]:
from pyspark.sql.functions import months_between, floor, to_date, concat, lit

for name in ["silver_train_df", "silver_test_df"]:
    df = globals()[name]
    df = (
        df.withColumn(
            "CompetitionStartDate",
            to_date(concat(col("CompetitionOpenSinceYear"), lit("-"), col("CompetitionOpenSinceMonth"), lit("-01")))
        )
        .withColumn("CompetitionAgeMonths", floor(months_between(col("Date"), col("CompetitionStartDate"))))
        .withColumn("CompetitionAgeMonths", when(col("CompetitionAgeMonths") < 0, 0).otherwise(col("CompetitionAgeMonths")))
    )
    globals()[name] = df

silver_train_df.select("Store", "Date", "CompetitionAgeMonths", "HasCompetitionInfo", "IsPromo2Active").show(10)

+-----+----------+--------------------+------------------+--------------+
|Store|      Date|CompetitionAgeMonths|HasCompetitionInfo|IsPromo2Active|
+-----+----------+--------------------+------------------+--------------+
|    1|2015-07-31|                  82|                 1|             0|
|    2|2015-07-31|                  92|                 1|             1|
|    3|2015-07-31|                 103|                 1|             1|
|    4|2015-07-31|                  70|                 1|             0|
|    5|2015-07-31|                   3|                 1|             0|
|    6|2015-07-31|                  19|                 1|             0|
|    7|2015-07-31|                  27|                 1|             0|
|    8|2015-07-31|                   9|                 1|             0|
|    9|2015-07-31|                 179|                 1|             0|
|   10|2015-07-31|                  70|                 1|             0|
+-----+----------+--------------------

## 9. Final validation

In [0]:
print("Duplicate (Store, Date) in silver_train:")
silver_train_df.groupBy("Store", "Date").agg(count("*").alias("c")).filter("c > 1").show()

print("Null check on key engineered columns:")
check_cols = ["HasCompetitionInfo", "CompetitionAgeMonths", "IsPromo2Active", "Year", "Month", "IsWeekend"]
silver_train_df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in check_cols]).show()

print("Final columns:", silver_train_df.columns)

Duplicate (Store, Date) in silver_train:
+-----+----+---+
|Store|Date|  c|
+-----+----+---+
+-----+----+---+

Null check on key engineered columns:
+------------------+--------------------+--------------+----+-----+---------+
|HasCompetitionInfo|CompetitionAgeMonths|IsPromo2Active|Year|Month|IsWeekend|
+------------------+--------------------+--------------+----+-----+---------+
|                 0|                   0|             0|   0|    0|        0|
+------------------+--------------------+--------------+----+-----+---------+

Final columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', '_ingested_at', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', '_ingested_at', 'HasCompetitionInfo', 'Year', 'Month', 'Week', 'Day', 'Quarter', 'IsWeekend', 'IsPromo2Active', 'CompetitionStartDate', 'CompetitionAgeMont

## 10. Write silver tables

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS retail_intelligence.silver")

silver_store_df.write.format("delta").mode("overwrite").saveAsTable("retail_intelligence.silver.silver_store")

# Drop duplicate _ingested_at columns from joined DataFrames before writing
silver_train_clean = silver_train_df.dropDuplicates(["Store", "Date"]).drop(silver_store_df["_ingested_at"])
silver_test_clean = silver_test_df.dropDuplicates(["Store", "Date"]).drop(silver_store_df["_ingested_at"])

silver_train_clean.write.format("delta").mode("overwrite").saveAsTable("retail_intelligence.silver.silver_train")
silver_test_clean.write.format("delta").mode("overwrite").saveAsTable("retail_intelligence.silver.silver_test")

print("Silver Store:", spark.table("retail_intelligence.silver.silver_store").count())
print("Silver Train:", spark.table("retail_intelligence.silver.silver_train").count())
print("Silver Test:", spark.table("retail_intelligence.silver.silver_test").count())

Silver Store: 1115
Silver Train: 1017209
Silver Test: 41088


In [0]:
display(spark.table("retail_intelligence.silver.silver_train").select(
    "Store", "Date", "Sales", "CompetitionAgeMonths", "IsPromo2Active", "IsWeekend", "HasCompetitionInfo"
))

Store,Date,Sales,CompetitionAgeMonths,IsPromo2Active,IsWeekend,HasCompetitionInfo
3,2015-07-31,8314,103,1,0,1
10,2015-07-31,7185,70,0,0,1
33,2015-07-31,10789,26,0,0,1
41,2015-07-31,6938,72,1,0,1
47,2015-07-31,9379,27,1,0,1
57,2015-07-31,11594,13,0,0,1
61,2015-07-31,5572,91,1,0,1
62,2015-07-31,7495,72,0,0,1
119,2015-07-31,7038,65,0,0,1
120,2015-07-31,10392,7,1,0,1
